# Step 4.2: test what the client's people told us (beliefs B1 to B9)

**The idea.** In interviews and voice notes, client staff said things like "schemes are not the problem". `notes/beliefs.md` lists these as B1 to B9.
Each one is a claim we can often check in the data. For each we give a **verdict**: *supported*, *not supported*, *partly*, or *cannot test*, with the number behind it.

**Read this first.** "Supported" means the data is consistent with what was said. It does not prove cause. Anything the data cannot show (payouts, forecast error, approvals) is marked *cannot test*, never guessed.

Run top to bottom. Each section ends with a **Check** line. Nothing here changes any data file.

In [11]:
import os, sys
import numpy as np, pandas as pd
sys.path.insert(0, os.path.join('..', 'pilot'))
from agent.data import load, fy_quarter
OUT = os.path.join('..', 'data-clean', 'audit', 'beliefs'); os.makedirs(OUT, exist_ok=True)
sku = load('sku_master').set_index('sku_code'); stk = load('stockist_master').set_index('stockist_code')
ps = load('primary_sales'); ec = load('expiry_claims'); sn = load('cnf_stock_snapshot'); bat = load('batch_master')
ps['q'] = fy_quarter(ps.invoice_date)
QS = ['PFL1001','PFL1002','PFL1003','PFL1101','PFL1102','PFL1103','PFL1105','PFL1202','PFL1301','PFL1303','PFL1004','PFL1104','PFL1201','PFL1302','PFL1401']  # Annexure B, tiers A and B
V = []   # verdict rows
def verdict(b, claim, v, number, source): V.append(dict(belief=b, claim=claim, verdict=v, evidence=number, source=source)); print(b, '->', v, '|', number)

## B1. "We are short and long at the same time" (CFO)
**Test.** On a snapshot date, a C&F has a stock-out marker for a SKU while another C&F holds more than 2 months of cover of the same SKU.
Cover = stock on hand divided by average monthly outward. (Lucknow stocks only 13 SKUs, so its marker may sometimes mean "not stocked"; we show results with and without Lucknow.)

In [12]:
h = sn[~sn.is_stockout_row].copy(); h['pts'] = h.sku_code.map(sku.pts_inr)
G = h.groupby(['snapshot_date','cnf_code','sku_code']).agg(q=('qty_on_hand_packs','sum'), avg=('avg_monthly_outward_packs','max')).reset_index()
G['cover'] = G.q / G.avg.replace(0, np.nan); G['pts'] = G.sku_code.map(sku.pts_inr)
so = sn[sn.is_stockout_row][['snapshot_date','cnf_code','sku_code']].drop_duplicates()
def pairs(so):
    m = so.merge(G[G.cover > 2][['snapshot_date','sku_code','cnf_code']].rename(columns={'cnf_code':'long_cnf'}), on=['snapshot_date','sku_code'])
    m = m[m.cnf_code != m.long_cnf]
    return m.groupby(['snapshot_date','sku_code']).size().reset_index(name='n')
all_sd = so.groupby(['snapshot_date','sku_code']).ngroups
p_all = pairs(so); p_noln = pairs(so[so.cnf_code != 'CF-N02'])
noln = so[so.cnf_code != 'CF-N02'].groupby(['snapshot_date','sku_code']).ngroups
print('all C&Fs      :', len(p_all), 'of', all_sd, 'SKU-dates short somewhere and long elsewhere')
print('without Lucknow:', len(p_noln), 'of', noln)
p_all.to_csv(os.path.join(OUT, 'b1_short_and_long.csv'), index=False)
verdict('B1', 'short and long at the same time', 'Supported',
        f'{len(p_all)} of {all_sd} SKU-dates ({len(p_noln)} of {noln} without Lucknow) have a stock-out at one C&F and >2 months cover at another',
        'cnf_stock_snapshot, sku_master')
assert len(p_noln) > 0
print('Check: B1 counted both with and without Lucknow; both positive.')

all C&Fs      : 100 of 137 SKU-dates short somewhere and long elsewhere
without Lucknow: 22 of 54
B1 -> Supported | 100 of 137 SKU-dates (22 of 54 without Lucknow) have a stock-out at one C&F and >2 months cover at another
Check: B1 counted both with and without Lucknow; both positive.


## B2. Last-10-days doubling and the "month-one cliff" (Kavita Rao)
**Test.** Share of a quarter's invoiced value that falls in the last 10 days and in each month, for quarter-scheme SKUs vs other SKUs.

In [13]:
s = ps[ps.batch_no.notna()].copy(); s['isqs'] = s.sku_code.isin(QS)
qend = s.invoice_date + pd.offsets.QuarterEnd(0)          # calendar quarter end (Mar/Jun/Sep/Dec)
# the client's FY quarters are Apr-Jun etc, i.e. calendar quarters; month-of-quarter below
s['dleft'] = (qend - s.invoice_date).dt.days; s['last10'] = s.dleft < 10
s['mq'] = ((s.invoice_date.dt.month - 1) % 3) + 1
def share(g): return pd.Series({'last10_%': round(g[g.last10].net_value_inr.sum()/g.net_value_inr.sum()*100,1),
                                 **{f'month{i}_%': round(g[g.mq==i].net_value_inr.sum()/g.net_value_inr.sum()*100,1) for i in (1,2,3)}})
r = s.groupby('isqs').apply(share); r.index = ['other SKUs','quarter-scheme SKUs']; print(r)
r.to_csv(os.path.join(OUT, 'b2_quarter_end.csv'))
a = r.loc['quarter-scheme SKUs']; o = r.loc['other SKUs']
verdict('B2', 'last-10-days doubling / month-one cliff', 'Supported',
        f'quarter-scheme SKUs: {a["last10_%"]}% in last 10 days vs {o["last10_%"]}% for other SKUs; month 1 {a["month1_%"]}% vs month 3 {a["month3_%"]}%',
        'primary_sales, 02 Annexure B')
assert a['last10_%'] > 2*o['last10_%']
print('Check: quarter-scheme share in last 10 days is more than double the others.')

                     last10_%  month1_%  month2_%  month3_%
other SKUs               11.2      33.1      33.3      33.5
quarter-scheme SKUs      39.4      17.4      30.7      51.8
B2 -> Supported | quarter-scheme SKUs: 39.4% in last 10 days vs 11.2% for other SKUs; month 1 17.4% vs month 3 51.8%
Check: quarter-scheme share in last 10 days is more than double the others.


## B3. "Schemes are not the problem" (Arjun Sethi)
**Test.** Match each non-rejected claim to the stockist and batch it came from. Compare claimed packs per pack invoiced for batches that were sold on a scheme vs batches that were not.
This shows whether claims **go with** schemes. It does not prove schemes cause them.

In [14]:
sup = ps[ps.batch_no.notna()].copy(); sup['packs'] = sup.invoiced_qty + sup.free_qty.fillna(0); sup['sch'] = sup.scheme_code.fillna('NA') != 'NO_SCHEME'
sb = sup.groupby(['stockist_code','batch_no']).agg(packs=('packs','sum'), sch=('sch','max')).reset_index()
c = ec[ec.status != 'Rejected'].merge(sb, on=['stockist_code','batch_no'], how='left')
tr = c[c.sch.notna()]
rate = (tr.groupby('sch').qty_claimed.sum() / sb.groupby('sch').packs.sum() * 100).round(3)
val = tr.groupby('sch').claim_value_derived_inr.sum()
print('claimed packs per 100 invoiced:', rate.to_dict()); print('claim value INR cr:', (val/1e7).round(2).to_dict())
print('traceable claims:', len(tr), 'of', len(c))
tr.groupby(['sch','claim_type']).size().unstack().to_csv(os.path.join(OUT, 'b3_claims_by_scheme.csv'))
verdict('B3', 'schemes are not the problem', 'Not supported',
        f'scheme batches {rate[True]}% of packs claimed vs {rate[False]}% non-scheme (~{rate[True]/rate[False]:.0f}x); Rs {val[True]/1e7:.2f} cr of Rs {val.sum()/1e7:.2f} cr traceable claim value; {len(tr)} of {len(c)} claims traceable',
        'primary_sales, expiry_claims (claim_value_derived_inr)')
assert rate[True] > 5 * rate[False]
print('Check: ratio computed on packs, value uses derived claim value.')

claimed packs per 100 invoiced: {False: 0.28, True: 2.711}
claim value INR cr: {False: 0.93, True: 6.37}
traceable claims: 3501 of 4472
B3 -> Not supported | scheme batches 2.711% of packs claimed vs 0.28% non-scheme (~10x); Rs 6.37 cr of Rs 7.31 cr traceable claim value; 3501 of 4472 claims traceable
Check: ratio computed on packs, value uses derived claim value.


## B4. Repeat claims, and breakage in Odisha and Assam (Neha Bansal)
**Test.** Repeat claims come from rule R4 (`step4_rules.ipynb`: 526 repeats, 98 probable duplicates). Breakage and short-supply claims by state, as a share of net primary value.

In [15]:
b = ec[ec.claim_type.isin(['Breakage in transit','Short supply'])].merge(stk[['state']], left_on='stockist_code', right_index=True)
purch = ps.merge(stk[['state']], left_on='stockist_code', right_index=True).groupby('state').net_value_inr.sum()
r = pd.DataFrame({'claims': b.groupby('state').size(), 'claim_inr': b.groupby('state').claim_value_derived_inr.sum()}).fillna(0)
r['pct_of_primary'] = (r.claim_inr / purch * 100).round(3); r['stockists'] = stk.groupby('state').size(); r = r.dropna().sort_values('pct_of_primary', ascending=False)
nat = b.claim_value_derived_inr.sum() / ps.net_value_inr.sum() * 100
print(r.head(8)); print('national %:', round(nat, 3)); r.to_csv(os.path.join(OUT, 'b4_breakage_by_state.csv'))
od, asm = r.loc['Odisha'], r.loc['Assam']
verdict('B4', 'breakage worst in Odisha and Assam', 'Weak',
        f'Odisha {od.pct_of_primary}% ({int(od.claims)} claims, {int(od.stockists)} stockists), Assam {asm.pct_of_primary}% ({int(asm.claims)} claims) vs national {nat:.3f}%; breakage+short is only {len(b)} claims',
        'expiry_claims, primary_sales, stockist_master')
verdict('B4b', 'repeat claims are common', 'Supported', '526 repeats, 98 probable duplicates (61 credited, Rs 11.1 lakh)', 'step4_rules.ipynb R4')
print('Check: states with 3 stockists give very small counts; do not over-read the ranking.')

               claims  claim_inr  pct_of_primary  stockists
state                                                      
Rajasthan           9    62316.4           0.066          3
Odisha              7    72742.1           0.061          3
Uttar Pradesh      77   384406.6           0.046         31
Puducherry         15    56018.5           0.044          5
Telangana          22   106480.6           0.040         10
Maharashtra        55   211570.8           0.031         21
West Bengal        42   150417.0           0.029         16
Assam               7    39276.4           0.028          3
national %: 0.031
B4 -> Weak | Odisha 0.061% (7 claims, 3 stockists), Assam 0.028% (7 claims) vs national 0.031%; breakage+short is only 374 claims
B4b -> Supported | 526 repeats, 98 probable duplicates (61 credited, Rs 11.1 lakh)
Check: states with 3 stockists give very small counts; do not over-read the ranking.


## B5. East: Kolkata primary has fallen
**Test.** Net primary value by C&F and quarter.

In [16]:
t = ps.pivot_table(index='q', columns='cnf_code', values='net_value_inr', aggfunc='sum').div(1e7).round(1); print(t)
t.to_csv(os.path.join(OUT, 'b5_primary_by_cnf_quarter.csv'))
first, last = t['CF-E01'].iloc[0], t['CF-E01'].iloc[-1]
verdict('B5', 'Kolkata primary has fallen', 'Supported', f'CF-E01 Rs {first} cr (first quarter) to Rs {last} cr (last), {((last/first)-1)*100:.0f}%; CF-N01/N02 rose',
        'primary_sales')
print('Check: last quarter FY27Q2 is complete to 30 Sep 2026 (latest invoice date).')

cnf_code  CF-E01  CF-N01  CF-N02  CF-S01  CF-S02  CF-W01
q                                                       
FY25Q1       8.7     7.8     7.2     8.5     7.1     7.0
FY25Q2       9.2     8.3     7.5     9.0     7.6     7.5
FY25Q3      10.4     9.8     8.6    10.4     8.8     8.5
FY25Q4       9.8     9.1     8.1     9.7     8.3     8.1
FY26Q1       9.6     8.8     8.1     9.7     8.2     8.0
FY26Q2       9.7     8.8     8.2     9.5     7.8     7.8
FY26Q3       8.5     8.4     8.6     9.3     8.1     7.5
FY26Q4       8.1     8.7     8.5     9.3     7.6     7.2
FY27Q1       7.8     9.5     9.4     8.9     8.0     7.7
FY27Q2       7.5     9.3     9.4     8.7     7.3     8.0
B5 -> Supported | CF-E01 Rs 8.7 cr (first quarter) to Rs 7.5 cr (last), -14%; CF-N01/N02 rose
Check: last quarter FY27Q2 is complete to 30 Sep 2026 (latest invoice date).


## B6 and B7. Forecast, allocation, and what happens to expiry
- **B6** forecast error (MAPE about 14%): there is no forecast file, so **cannot test**. "Allocation is static" is tested together with B2.
- **B7** payout is zero below 100% and the company takes expiry back: payout rules are not in the data, so **cannot test**. What we can test is whether stockists with more scheme value also claim more.

In [17]:
s2 = sup[sup.sch].groupby('stockist_code').net_value_inr.sum()
cl = ec[ec.status != 'Rejected'].groupby('stockist_code').claim_value_derived_inr.sum()
d = pd.DataFrame({'scheme_inr': s2, 'claim_inr': cl}).fillna(0)
corr = d.scheme_inr.corr(d.claim_inr)
print('correlation across', len(d), 'stockists:', round(corr, 2))
verdict('B6', 'forecast error about 14%', 'Cannot test', 'no forecast file in the pack', '04 / data-pack inventory')
verdict('B7', 'stockists load up on schemes then claim expiry', 'Partly', f'stockist-level correlation of scheme value and claim value = {corr:.2f}; payout and take-back terms not in data', 'primary_sales, expiry_claims')
g = stk[stk.stockist_name.str.contains('Interpharm', na=False)]
verdict('B7b', 'New Interpharm Guntur, about Rs 8 lakh in last week of Sep', 'Cannot test', f'no stockist named Interpharm in stockist_master (matches: {len(g)}); 5 Guntur stockists exist, none shows a clear spike', 'stockist_master, primary_sales')
assert len(g) == 0
print('Check: name not found, so we do not pick a different stockist.')

correlation across 156 stockists: 0.83
B6 -> Cannot test | no forecast file in the pack
B7 -> Partly | stockist-level correlation of scheme value and claim value = 0.83; payout and take-back terms not in data
B7b -> Cannot test | no stockist named Interpharm in stockist_master (matches: 0); 5 Guntur stockists exist, none shows a clear spike
Check: name not found, so we do not pick a different stockist.


## B8. Stock-outs of Amoxclav (PFL1002) and Azithromycin (PFL1001), and Ganesh Pharma Traders
**B8a test.** Share of order lines not supplied (blank batch) for each SKU and C&F from April 2026.
**B8b test.** STK0007 Ganesh Pharma Traders (Punjab, CF-N01), Monte LC (PFL1301): monthly invoiced and free packs.

In [18]:
x = ps[ps.ordered_qty.notna()].copy(); x['ns'] = x.batch_no.isna(); x27 = x[x.invoice_date >= '2026-04-01']
base = x27.groupby('cnf_code').ns.mean().mul(100).round(1)
tab = pd.DataFrame({s: x27[x27.sku_code == s].groupby('cnf_code').ns.mean().mul(100).round(1) for s in ['PFL1002','PFL1001']}); tab['all SKUs'] = base
print(tab); tab.to_csv(os.path.join(OUT, 'b8a_not_supplied_fy27.csv'))
g, l, h = 'CF-N01', 'CF-N02', 'CF-S02'
verdict('B8a', 'Amoxclav/Azithro stock-outs at Gurugram, Lucknow, Hyderabad', 'Supported at Gurugram and Lucknow; weaker at Hyderabad',
        f'Order lines not supplied since Apr 2026 (all SKUs in brackets): Amoxclav Gurugram {tab.loc[g,"PFL1002"]}% ({base[g]}), Lucknow {tab.loc[l,"PFL1002"]}% ({base[l]}), Hyderabad {tab.loc[h,"PFL1002"]}% ({base[h]}); Azithromycin Gurugram {tab.loc[g,"PFL1001"]}%, Lucknow {tab.loc[l,"PFL1001"]}%, Hyderabad {tab.loc[h,"PFL1001"]}%. Hyderabad is above its own normal but no worse than Chennai or Bhiwandi',
        'primary_sales (from Apr 2026)')
p = ps[(ps.stockist_code == 'STK0007') & (ps.sku_code == 'PFL1301') & ps.batch_no.notna()]
m = p.groupby(p.invoice_date.dt.to_period('M')).agg(inv=('invoiced_qty','sum'), free=('free_qty','sum')); print(m.tail(10))
avg = m.inv.iloc[:-1].mean(); sep = m.iloc[-1]
verdict('B8b', 'Ganesh Pharma Traders ~1,000 strips + ~150 free vs 500-600/month', 'Supported',
        f'Sep 2026: {int(sep.inv)} invoiced + {int(sep.free)} free vs average {avg:.0f}/month before; same pattern at each quarter end (Mar, Jun)', 'primary_sales')
assert stk.loc['STK0007','stockist_name'] == 'Ganesh Pharma Traders'
print('Check: stockist code and name confirmed from stockist_master.')

          PFL1002  PFL1001  all SKUs
cnf_code                            
CF-E01        3.3      7.4       2.7
CF-N01       33.3     30.3       7.8
CF-N02       50.0     33.7      15.7
CF-S01       11.7     18.6       7.7
CF-S02       14.2     11.6       6.1
CF-W01       14.0      9.2       4.8
B8a -> Supported at Gurugram and Lucknow; weaker at Hyderabad | Order lines not supplied since Apr 2026 (all SKUs in brackets): Amoxclav Gurugram 33.3% (7.8), Lucknow 50.0% (15.7), Hyderabad 14.2% (6.1); Azithromycin Gurugram 30.3%, Lucknow 33.7%, Hyderabad 11.6%. Hyderabad is above its own normal but no worse than Chennai or Bhiwandi
               inv  free
invoice_date            
2025-12        403     0
2026-01        418     0
2026-02        608     0
2026-03        932   138
2026-04        226     0
2026-05        550     0
2026-06        927   137
2026-07        459     0
2026-08        551     0
2026-09       1000   148
B8b -> Supported | Sep 2026: 1000 invoiced + 148 free vs average 57

## B9. Kolkata: batch rule not live, old cardio/pain lots, godown full
**Test.** At the latest snapshot: how many SKUs hold more than one batch at each C&F, and the age of stock at manufacture date. (The snapshot has no stock within 180 days of expiry, so we cannot show near-expiry stock.)

In [19]:
l = sn[(sn.snapshot_date == sn.snapshot_date.max()) & ~sn.is_stockout_row].merge(bat[['batch_no','mfg_date']], on='batch_no', how='left')
l['age'] = (l.snapshot_date - l.mfg_date).dt.days; l['val'] = l.qty_on_hand_packs * l.sku_code.map(sku.pts_inr)
multi = l.groupby('cnf_code').apply(lambda d: (d.groupby('sku_code').batch_no.nunique() > 1).mean()).round(2)
t = l.groupby('cnf_code').agg(value_cr=('val', lambda x: round(x.sum()/1e7, 1)), median_age_days=('age', 'median')); t['skus_with_2plus_batches'] = multi
print(t); t.to_csv(os.path.join(OUT, 'b9_latest_snapshot_by_cnf.csv'))
k = l[l.cnf_code == 'CF-E01'].merge(sku[['therapy_area']], left_on='sku_code', right_index=True).groupby('therapy_area').agg(value_cr=('val', lambda x: round(x.sum()/1e7, 2)), median_age=('age','median')); print(k)
verdict('B9', 'Hooghly batch rule not live; old stock; godown full', 'Partly',
        f'Kolkata holds the most stock (Rs {t.loc["CF-E01","value_cr"]} cr), oldest (median {t.loc["CF-E01","median_age_days"]:.0f} days vs ~200), and every SKU holds 2+ batches ({multi["CF-E01"]:.0%} vs {multi.drop("CF-E01").min():.0%} to {multi.drop("CF-E01").max():.0%} elsewhere); godown fullness % not in data',
        'cnf_stock_snapshot, batch_master')
print('Check: mfg_date comes from batch_master (join on batch_no).')

          value_cr  median_age_days  skus_with_2plus_batches
cnf_code                                                    
CF-E01         3.9            294.0                     1.00
CF-N01         1.7            192.0                     0.10
CF-N02         0.4            222.0                     0.00
CF-S01         2.3            227.0                     0.41
CF-S02         2.2            227.5                     0.55
CF-W01         2.4            212.0                     0.29
                        value_cr  median_age
therapy_area                                
Anti-infectives             0.58       225.0
Cardio-metabolic            1.01       346.0
Central nervous system      0.12       337.0
Gastro                      0.74       314.0
Pain and inflammation       0.48       319.0
Respiratory                 0.07       301.0
Vitamins and nutrition      0.91       165.0
B9 -> Partly | Kolkata holds the most stock (Rs 3.9 cr), oldest (median 294 days vs ~200), and every SKU ho

## Summary

In [20]:
v = pd.DataFrame(V); v.to_csv(os.path.join(OUT, 'beliefs_summary.csv'), index=False)
pd.set_option('display.max_colwidth', 120); print(v[['belief','verdict','evidence']].to_string(index=False))
assert len(v) >= 10 and v.verdict.str.startswith(('Supported','Not supported','Partly','Weak','Cannot test')).all()
print('\nALL BELIEF CHECKS PASSED; wrote', os.path.join(OUT, 'beliefs_summary.csv'))

belief                                                verdict                                                                                                                                                                                                                                                                         evidence
    B1                                              Supported                                                                                                                                                                       100 of 137 SKU-dates (22 of 54 without Lucknow) have a stock-out at one C&F and >2 months cover at another
    B2                                              Supported                                                                                                                                                                               quarter-scheme SKUs: 39.4% in last 10 days vs 11.2% for other SKUs; month 1 17.4% vs month 3 5